# PPO from Scratch

Proximal Policy Optimization is the algorithm behind RLHF, and it is best understood as
a sequence of fixes to one underlying problem: **the policy gradient is correct in
expectation and almost unusable in practice**. It has enormous variance, and it is
on-policy, so every sample is thrown away after one gradient step.

PPO's answer is to reuse each batch for several epochs while preventing the policy from
moving too far — using a clipped objective that is crude, theoretically inelegant, and
works remarkably well.

This notebook builds it up in that order: the estimator, the variance fix, the clip, then
a working implementation. First topic in the
[RL & Training Dynamics](grpo-rlvr.ipynb) track.

## 1. What & Why

The policy gradient theorem says:

```
∇J(θ) = E[ ∇log π_θ(a|s) · A(s,a) ]
```

Sample actions, compute the log-probability gradient, weight by how good the action was.
Correct, unbiased, and beset by three practical problems:

1. **Variance.** Weighting by raw return means a trajectory's noise dominates the signal.
   Subtracting a baseline — using the *advantage* rather than the return — is the
   standard fix and it is dramatic.
2. **On-policy waste.** The expectation is over the *current* policy, so after one update
   the samples are stale. For LLM RLHF, where sampling is the expensive part, throwing
   away a batch after one step is unaffordable.
3. **Destructive updates.** A large step can collapse the policy to something that
   generates garbage, and because the next batch is sampled *from that policy*, there is
   no way back. Supervised learning has no equivalent failure — a bad step there still
   sees the same dataset next time.

PPO addresses (2) and (3) together with importance sampling plus a clip: reuse the batch
for `K` epochs, but ignore any gradient that would push the policy ratio outside
`[1−ε, 1+ε]`.

**In RLHF specifically** the reward comes from a
[reward model](reward-models-and-hacking.ipynb) and a KL penalty to the reference policy
is added, because the reward model is only valid near the distribution it was trained on.

## 2. Mental Model

**A leash, not a step size.**

The intuition PPO is built on: you cannot trust your gradient estimate far from where you
sampled. The samples were drawn from `π_old`; as the policy moves, the importance ratio
`r = π_new/π_old` drifts from 1 and the estimate degrades. Beyond some distance the
gradient is not merely noisy — it is measuring the wrong distribution.

TRPO enforced this with a hard KL constraint and a second-order solve. PPO replaces that
with a leash: **clip the objective so that once the ratio leaves `[1−ε, 1+ε]`, moving
further gives no additional reward.** The gradient becomes zero out there, so the
optimiser stops pulling.

The asymmetry is the part worth internalising, and it is not obvious:

- For a **good** action (`A > 0`), the clip caps the upside — you stop being rewarded for
  making it much more likely.
- For a **bad** action (`A < 0`), the clip caps how much you are rewarded for suppressing
  it — but if the ratio has already fallen below `1−ε`, the objective is *unclipped* and
  the gradient still flows.

So the leash is tight on making things more likely and loose on making them less likely.
That asymmetry is deliberate: recovering from a bad action should always be possible.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Policy gradient** | `∇J = E[∇log π(a\|s) · A]`. Unbiased, high variance. |
| **Return `G`** | Discounted sum of future rewards from a state. |
| **Baseline** | Any function of state subtracted from the return. Reduces variance without adding bias. |
| **Advantage `A`** | `Q(s,a) − V(s)` — how much better this action was than the state's average. The baseline made concrete. |
| **Critic / value head** | The learned `V(s)`. In LLM RLHF, usually a second head on the same backbone. |
| **GAE(λ)** | Generalised Advantage Estimation: an exponentially-weighted blend of n-step advantages. `λ` trades bias for variance. |
| **Importance ratio `r`** | `π_new(a\|s) / π_old(a\|s)`. Corrects for the samples being off-policy after the first epoch. |
| **Clipped surrogate** | `min(r·A, clip(r, 1−ε, 1+ε)·A)`. PPO's objective; `ε` is typically 0.1–0.2. |
| **KL penalty** | In RLHF, a penalty against drifting from the reference (SFT) policy. Keeps the reward model in-distribution. |
| **Epochs per batch `K`** | How many passes over one rollout. The reuse the clip makes safe. Usually 1–4. |
| **Entropy bonus** | A term encouraging a less-peaked policy, to preserve exploration. |

## 4. Setup

NumPy. Every component is implemented and tested on problems small enough to verify by
hand, which is the point — PPO is a stack of simple pieces, and the difficulty is entirely
in how they interact.

In [1]:
# %pip install numpy

import numpy as np

rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — the baseline, and how much variance it removes

The single most important variance reduction in policy gradients, and it costs nothing in
bias.

In [2]:
# A 4-armed bandit. True mean rewards, all POSITIVE -- which is what makes a baseline
# matter: without one, every action is reinforced, just by different amounts.
true_rewards = np.array([10.0, 11.0, 12.0, 13.0])
K_ARMS = len(true_rewards)

def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

def grad_estimates(logits, n_samples, baseline_mode, seed):
    '''Return n_samples independent single-sample gradient estimates for arm 0's logit.'''
    r = np.random.default_rng(seed)
    p = softmax(logits)
    out = []
    for _ in range(n_samples):
        a = r.choice(K_ARMS, p=p)
        reward = true_rewards[a] + r.normal(0, 1.0)
        if baseline_mode == "none":
            weight = reward
        elif baseline_mode == "mean":
            weight = reward - float(true_rewards @ p)      # V(s), the expected reward
        # d log pi(a) / d logit_0  =  1[a==0] - p[0]
        out.append(weight * ((1.0 if a == 0 else 0.0) - p[0]))
    return np.array(out)

logits = np.zeros(K_ARMS)
print(f"{'baseline':10} {'mean estimate':>15} {'std':>10} {'variance ratio':>16}")
base_var = None
for mode in ("none", "mean"):
    est = grad_estimates(logits, 20000, mode, seed=1)
    if base_var is None:
        base_var = est.var()
    print(f"{mode:10} {est.mean():15.4f} {est.std():10.4f} {est.var()/base_var:16.3f}")

print("\nBoth estimators point the same way -- subtracting a baseline does not bias the")
print("gradient, because E[baseline * dlog pi] = 0. But the variance falls sharply.")
print("\nThe reason is visible in the setup: with all rewards positive and no baseline,")
print("EVERY sampled action gets reinforced. The learning signal is the small difference")
print("between rewards, buried under a large common term. The baseline subtracts that")
print("common term, which is all 'advantage' means.")

baseline     mean estimate        std   variance ratio
none               -0.4239     4.5313            1.000


mean               -0.3797     0.6445            0.020

Both estimators point the same way -- subtracting a baseline does not bias the
gradient, because E[baseline * dlog pi] = 0. But the variance falls sharply.

The reason is visible in the setup: with all rewards positive and no baseline,
EVERY sampled action gets reinforced. The learning signal is the small difference
between rewards, buried under a large common term. The baseline subtracts that
common term, which is all 'advantage' means.


### Example 2 — the clipped objective, and its asymmetry

Evaluate PPO's surrogate as a function of the policy ratio, for a good action and a bad
one. The shape explains the algorithm's behaviour better than the formula does.

In [3]:
EPS = 0.2

def ppo_objective(ratio, advantage, eps=EPS):
    unclipped = ratio * advantage
    clipped = np.clip(ratio, 1 - eps, 1 + eps) * advantage
    return np.minimum(unclipped, clipped)

ratios = np.array([0.5, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4, 2.0])
print(f"eps = {EPS}\n")
print(f"{'ratio':>7} {'A=+1 objective':>16} {'gradient?':>11} | "
      f"{'A=-1 objective':>16} {'gradient?':>11}")
for r in ratios:
    pos = ppo_objective(r, +1.0)
    neg = ppo_objective(r, -1.0)
    # a flat objective means zero gradient
    d = 1e-6
    g_pos = abs(ppo_objective(r + d, +1.0) - pos) > 1e-12
    g_neg = abs(ppo_objective(r + d, -1.0) - neg) > 1e-12
    print(f"{r:7.2f} {pos:16.3f} {str(g_pos):>11} | {neg:16.3f} {str(g_neg):>11}")

print("\nFor a GOOD action (A>0): the objective stops rising above ratio 1.2. Beyond the")
print("leash there is nothing to gain, so the gradient vanishes and the policy stops")
print("pushing that action's probability up.")
print("\nFor a BAD action (A<0): the objective is flat ABOVE 0.8 -- but keeps improving")
print("as the ratio falls below it. Suppressing a bad action is never cut off.")
print("\nThat asymmetry is the design. The clip restrains enthusiasm, not correction.")

eps = 0.2

  ratio   A=+1 objective   gradient? |   A=-1 objective   gradient?
   0.50            0.500        True |           -0.800       False
   0.70            0.700        True |           -0.800       False
   0.80            0.800        True |           -0.800        True
   0.90            0.900        True |           -0.900        True
   1.00            1.000        True |           -1.000        True
   1.10            1.100        True |           -1.100        True
   1.20            1.200       False |           -1.200        True
   1.40            1.200       False |           -1.400        True
   2.00            1.200       False |           -2.000        True

For a GOOD action (A>0): the objective stops rising above ratio 1.2. Beyond the
leash there is nothing to gain, so the gradient vanishes and the policy stops
pushing that action's probability up.

For a BAD action (A<0): the objective is flat ABOVE 0.8 -- but keeps improving
as the ratio falls below it. Sup

### Example 3 — GAE: trading bias against variance

The advantage estimate itself is a choice. `λ=0` uses the one-step TD error (low variance,
biased by the critic's errors); `λ=1` uses the full Monte-Carlo return (unbiased, high
variance). Everything in between interpolates.

In [4]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    '''Generalised Advantage Estimation, computed backwards in one pass.'''
    T = len(rewards)
    adv = np.zeros(T)
    last = 0.0
    for t in reversed(range(T)):
        next_v = values[t + 1] if t + 1 < len(values) else 0.0
        delta = rewards[t] + gamma * next_v - values[t]
        last = delta + gamma * lam * last
        adv[t] = last
    return adv

T = 40
true_values = np.linspace(10, 0, T + 1)          # the true V(s) for this trajectory
GAMMA = 0.99

def one_trial(critic_scale, critic_noise, seed):
    '''A critic that OVERESTIMATES proportionally -- a state-dependent error.

    This matters: a constant offset largely cancels in the TD difference
    V(s') - V(s), so it barely biases low-lambda estimates. Real critics are
    wrong in state-dependent ways, which is the case worth simulating.
    '''
    r = np.random.default_rng(seed)
    values = true_values * critic_scale + r.normal(0, critic_noise, T + 1)
    rewards = np.diff(-true_values) + r.normal(0, 0.5, T)
    ret = sum((GAMMA ** t) * rewards[t] for t in range(T))
    truth = ret - true_values[0]                  # the advantage we are trying to estimate
    return {lam: compute_gae(rewards, values, lam=lam)[0]
            for lam in (0.0, 0.5, 0.95, 1.0)}, truth

trials = [one_trial(1.4, 1.0, s) for s in range(2000)]
truth = np.array([t for _, t in trials])

print("advantage estimate for the first step, over 2000 trajectories")
print("(critic OVERESTIMATES by 40%, and is noisy):\n")
print(f"{'lambda':>8} {'mean est':>10} {'bias':>9} {'std':>9} {'RMSE':>9}")
for lam in (0.0, 0.5, 0.95, 1.0):
    v = np.array([d[lam] for d, _ in trials])
    print(f"{lam:8.2f} {v.mean():10.3f} {v.mean() - truth.mean():9.3f} "
          f"{v.std():9.3f} {np.sqrt(np.mean((v - truth) ** 2)):9.3f}")
print(f"\n{'true advantage':>8}: mean {truth.mean():.3f}")

print("\nThe VARIANCE column is the unambiguous part, and it rises monotonically with")
print("lambda: more of the estimate comes from sampled rewards and less from the")
print("critic, so there is more noise. That direction always holds.")
print("\nThe BIAS column is more interesting than the textbook summary suggests. Each")
print("extreme inherits a DIFFERENT piece of the critic's error:")
print("  lambda=0 is the one-step TD error, so it inherits the critic's step-to-step")
print("           INCONSISTENCY -- how wrong V(s\') is relative to V(s).")
print("  lambda=1 is the Monte-Carlo return minus V(s0), so it inherits the critic's")
print("           error AT THE START STATE, in full.")
print("Here the critic overestimates proportionally, so its start-state error is large")
print("and lambda=1 carries all of it -- which is why lambda=1 has the LARGEST bias in")
print("this run, not the smallest. A differently-shaped critic error reverses that.")
print("\nWhat survives regardless is the shape of the RMSE column: both extremes are")
print("worse than the middle, and lambda around 0.95 is the standard default because")
print("it keeps most of the variance reduction while limiting exposure to whichever")
print("kind of critic error you happen to have.")
print("\n(A caveat for the policy gradient specifically: an error in the baseline that")
print("depends only on the STATE adds variance but not bias to the gradient, because")
print("E[b(s) * dlog pi] = 0. The bias measured here is bias in the ADVANTAGE")
print("ESTIMATE, which is what GAE is judged on and what the critic loss sees.)")

advantage estimate for the first step, over 2000 trajectories
(critic OVERESTIMATES by 40%, and is noisy):

  lambda   mean est      bias       std      RMSE
    0.00     -0.302     1.411     1.478     3.352
    0.50     -0.508     1.205     1.265     3.079
    0.95     -3.004    -1.291     1.776     2.265
    1.00     -5.732    -4.018     2.891     4.193

true advantage: mean -1.713

The VARIANCE column is the unambiguous part, and it rises monotonically with
lambda: more of the estimate comes from sampled rewards and less from the
critic, so there is more noise. That direction always holds.

The BIAS column is more interesting than the textbook summary suggests. Each
extreme inherits a DIFFERENT piece of the critic's error:
  lambda=0 is the one-step TD error, so it inherits the critic's step-to-step
           INCONSISTENCY -- how wrong V(s') is relative to V(s).
  lambda=1 is the Monte-Carlo return minus V(s0), so it inherits the critic's
           error AT THE START STATE, in ful

### Example 4 — a complete PPO loop, and what the clip is actually preventing

A contextual bandit — enough structure to need a policy and a critic, small enough to
verify. Then the same run with clipping disabled.

In [5]:
N_STATES, N_ACTIONS = 6, 4
# The optimal action differs per state; rewards are NOISY, so advantage estimates
# are frequently wrong -- which is the condition the clip exists to survive.
payoff = rng.normal(0, 1, (N_STATES, N_ACTIONS))
payoff[np.arange(N_STATES), rng.integers(0, N_ACTIONS, N_STATES)] += 3.0

def rollout(theta, n, seed):
    r = np.random.default_rng(seed)
    states = r.integers(0, N_STATES, n)
    probs = np.array([softmax(theta[s]) for s in states])
    actions = np.array([r.choice(N_ACTIONS, p=p) for p in probs])
    rewards = payoff[states, actions] + r.normal(0, 2.0, n)
    logp_old = np.log(probs[np.arange(n), actions] + 1e-12)
    return states, actions, rewards, logp_old

def kl_between(p_old, p_new):
    return float(np.sum(p_new * np.log((p_new + 1e-12) / (p_old + 1e-12))))

def train(clip=True, eps=0.2, epochs=10, iters=120, lr=1.0, batch=64, seed=0):
    theta = np.zeros((N_STATES, N_ACTIONS))
    critic = np.zeros(N_STATES)
    reward_hist, kls, max_ratios = [], [], []
    for it in range(iters):
        s, a, rew, logp_old = rollout(theta, batch, seed=1000 + it)
        old_policy = np.array([softmax(theta[x]) for x in range(N_STATES)])
        for st in range(N_STATES):                       # simple running critic
            m = s == st
            if m.any():
                critic[st] += 0.3 * (rew[m].mean() - critic[st])
        adv = rew - critic[s]
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)    # PPO standardises advantages
        for _ in range(epochs):                          # REUSE the batch
            grad = np.zeros_like(theta)
            for i in range(len(s)):
                p = softmax(theta[s[i]])
                ratio = np.exp(np.log(p[a[i]] + 1e-12) - logp_old[i])
                if clip and ((adv[i] > 0 and ratio > 1 + eps)
                             or (adv[i] < 0 and ratio < 1 - eps)):
                    continue                             # clipped: no gradient
                onehot = np.zeros(N_ACTIONS); onehot[a[i]] = 1.0
                grad[s[i]] += ratio * adv[i] * (onehot - p)
            theta += lr * grad / len(s)
        new_policy = np.array([softmax(theta[x]) for x in range(N_STATES)])
        kls.append(np.mean([kl_between(old_policy[x], new_policy[x])
                            for x in range(N_STATES)]))
        ratios = [np.exp(np.log(softmax(theta[s[i]])[a[i]] + 1e-12) - logp_old[i])
                  for i in range(len(s))]
        max_ratios.append(float(np.max(ratios)))
        reward_hist.append(float(payoff[s, a].mean()))
    return theta, reward_hist, kls, max_ratios

best = payoff.max(axis=1).mean()
print(f"best achievable mean reward: {best:.3f}   (10 epochs per batch, noisy rewards)\n")
print(f"{'setting':12} {'final reward':>13} {'mean KL/update':>15} {'max KL':>9} "
      f"{'max ratio':>10}")
for label, use_clip in (("clipped", True), ("unclipped", False)):
    _, hist, kls, mr = train(clip=use_clip)
    print(f"{label:12} {np.mean(hist[-20:]):13.3f} {np.mean(kls):15.4f} "
          f"{max(kls):9.4f} {max(mr):10.2f}")

print("\nRead the last two columns, not the first. Both runs solve this problem -- it")
print("is forgiving, and the clip is not there to make learning faster.")
print("\nWhat the clip actually does is bound POLICY MOVEMENT per update. Unclipped,")
print("the importance ratio reaches far outside [0.8, 1.2], meaning the tenth epoch is")
print("computing gradients from samples that no longer describe the current policy.")
print("The clipped run keeps the ratio inside its leash and roughly halves the KL")
print("travelled per update.")
print("\nOn a bandit that costs nothing. In LLM RLHF, where an over-large update can")
print("collapse the policy into degenerate text and every subsequent batch is sampled")
print("FROM that broken policy, it is the difference between a run and a write-off.")

best achievable mean reward: 3.172   (10 epochs per batch, noisy rewards)

setting       final reward  mean KL/update    max KL  max ratio


clipped              3.117          0.0012    0.0227       1.36


unclipped            3.134          0.0026    0.0658       4.59

Read the last two columns, not the first. Both runs solve this problem -- it
is forgiving, and the clip is not there to make learning faster.

What the clip actually does is bound POLICY MOVEMENT per update. Unclipped,
the importance ratio reaches far outside [0.8, 1.2], meaning the tenth epoch is
computing gradients from samples that no longer describe the current policy.
The clipped run keeps the ratio inside its leash and roughly halves the KL
travelled per update.

On a bandit that costs nothing. In LLM RLHF, where an over-large update can
collapse the policy into degenerate text and every subsequent batch is sampled
FROM that broken policy, it is the difference between a run and a write-off.


## 6. Gotchas & Pitfalls

- **Forgetting to normalise advantages.** PPO implementations almost universally
  standardise the advantage within a batch. Without it, `ε` means something different at
  every reward scale and the clip stops being a meaningful leash.
- **Reusing a batch too many times.** The clip bounds the damage, but the samples still
  become stale. `K = 1–4` is standard; large `K` with a loose clip is asking for the
  Example 4 failure.
- **A critic that is worse than useless.** GAE with `λ` near 0 leans hard on the critic
  (Example 3). If the critic is badly biased, a higher `λ` — or a critic-free method like
  [GRPO](grpo-rlvr.ipynb) — is the better answer.
- **Omitting the KL penalty in RLHF.** The reward model is only valid near the
  distribution it was trained on. Without a KL term the policy walks somewhere the reward
  model scores highly and humans do not — see
  [Reward Models & Reward Hacking](reward-models-and-hacking.ipynb).
- **Clipping the value loss without thinking.** Many implementations clip it by analogy
  to the policy loss. The theoretical justification is much weaker, and it sometimes
  hurts.
- **Entropy collapse.** Without an entropy bonus, the policy can become deterministic
  early, at which point exploration stops and the run is finished — but the loss curve
  looks fine.
- **Reward scale drift.** PPO is not scale-invariant. A reward whose magnitude changes
  during training silently changes the effective learning rate.
- **Assuming the clip is a trust region.** It is a heuristic approximation to one. It
  bounds the *ratio per sample*, not the KL divergence of the policy, and the two can come
  apart.

## 7. When to Use vs Alternatives

| Situation | Reach for |
|---|---|
| RLHF with a learned reward model, general-purpose | **PPO** — the well-understood default |
| Verifiable rewards (maths, code, tests) | [**GRPO / RLVR**](grpo-rlvr.ipynb) — no critic to train, much simpler |
| Preference data, no RL loop wanted | **DPO** — closed-form, no sampling; usually weaker than a well-tuned PPO but far easier |
| Very expensive sampling | **PPO** with several epochs per batch — precisely what the clip buys |
| A great critic already available | **A2C / vanilla PG** — the clip matters less if you can afford on-policy updates |
| Provable monotonic improvement | **TRPO** — the hard-constraint ancestor. Better guarantees, much heavier to implement |

**The honest position.** PPO is a pile of practical heuristics — clipping, advantage
normalisation, entropy bonuses, value-loss clipping, GAE — and reproductions show that the
implementation details account for much of its reported advantage over simpler baselines.
That is a fair criticism, and it is also why it works: those details are the accumulated
fixes for real failure modes.

For LLM post-training, PPO's position has narrowed. Where rewards are *verifiable*, the
simpler critic-free methods in [GRPO & Verifiable Rewards](grpo-rlvr.ipynb) are now often
preferred — no critic to train, no critic to be wrong. PPO remains the default where the
reward is a learned model and the value function genuinely helps.

## 8. Resources

- [Proximal Policy Optimization Algorithms](https://arxiv.org/abs/1707.06347) — Schulman et al., 2017. Short, and the clipped objective is derived in two pages.
- [High-Dimensional Continuous Control Using Generalized Advantage Estimation](https://arxiv.org/abs/1506.02438) — the GAE(λ) of Example 3.
- [Trust Region Policy Optimization](https://arxiv.org/abs/1502.05477) — the hard-constraint predecessor PPO approximates.
- [The 37 Implementation Details of Proximal Policy Optimization](https://iclr-blog-track.github.io/2022/03/25/ppo-implementation-details/) — indispensable if you are writing your own; every heuristic above, with ablations.
- [Implementation Matters in Deep RL: A Case Study on PPO and TRPO](https://arxiv.org/abs/2005.12729) — the evidence that the details, not the clip, carry much of the performance.
- [Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155) — InstructGPT; PPO as applied to LLMs, including the KL penalty.
- [TRL's PPOTrainer](https://huggingface.co/docs/trl/ppo_trainer) — a readable production implementation; see also [TRL](../03-llm-inference-training-optimization/trl-rlhf-dpo.ipynb).